# Flora v3 — selezione della flora siciliana e training PlantCLEF

**Le 40 classi di v2 non coprono la flora siciliana.** Questa versione genera le classi da una checklist regionale
versionata e le incrocia con il tuo CSV PlantCLEF; non presume che il dataset contenga tutte le specie.

## Ornamentali e coltivate: modalità estesa
Il default `CLASS_SCOPE='sicilia_plus_dataset'` include anche le specie utilizzabili del CSV che non risultano
nella checklist siciliana. Questo evita di escludere a priori una possibile ornamentale, ma **non dimostra
che ogni classe aggiuntiva sia coltivata in Sicilia** e non copre le specie assenti da PlantCLEF.
Il modello è quindi un catalogo PlantCLEF ampliato, con un riferimento regionale documentato; non viene
applicata automaticamente una priorità geografica alle probabilità. Per un modello più piccolo e mirato serve
una lista verificata delle ornamentali d'interesse, da aggiungere alle spontanee.
`CLASS_SCOPE='spontaneous_only'` limita invece i target alla checklist regionale.

## Avvio
1. Seleziona un runtime Colab GPU (l'audit iniziale funziona anche su CPU).
2. Mantieni il CSV in MyDrive o modifica `METADATA_CSV` nella cella 2.
3. Esegui: con `TRAINING_ENABLED=False` vengono prodotti inventario e report senza scaricare immagini né allenare.
4. Leggi `taxonomy/coverage.csv`: `trainable`, `insufficient_observations`, `no_usable_match`.
   Le ultime due categorie NON sono classi che il modello saprà riconoscere.
5. Per allenare il sottoinsieme disponibile, imposta `TRAINING_ENABLED=True`, mantenendo il NUOVO RUN_ID v3,
   e riesegui. Con `REQUIRE_FULL_REFERENCE_COVERAGE=True` il training si blocca se manca anche una specie di riferimento.
6. Baseline, teacher e distillazione restano percorsi separati e riprendibili. Non riusare i checkpoint v2 a 40 classi.

## Fonte e perimetro
La sorgente è **FlorItaly, Checklist 27 aprile 2023**, distribuita nel pacchetto `gibedini/floritaly`
(Gianni Bedini, Matteo Conti), versione Git fissata e checksum SHA256 verificato.
Non è una dichiarazione di completezza tassonomica al 2026: il portale FlorItaly può avere aggiornamenti successivi.
Si selezionano i taxa con indicazione di presenza `P` nella colonna `SIC`, compresi i record parentali `(P)`;
si raggruppano sottospecie e varietà **a livello di specie**. Gli ibridi con nome vengono mantenuti distinti.
Le sezioni di Taraxacum non sono specie e sono riportate separatamente, non trattate come classi.

Perimetro predefinito: **piante vascolari spontanee, autoctone e introdotte**, incluse le aliene casuali
(`INCLUDE_CASUAL_ALIENS=False` per escluderle). Non è l'elenco di tutte le ornamentali coltivate nei giardini,
né di muschi, alghe o licheni. Per le sole coltivate occorre una lista aggiuntiva verificata.
Le presenze dubbie, storiche non riconfermate, errate o estinte non vengono incluse automaticamente.

## Cosa significa “tutte le classi”
Si salva l'intero elenco di riferimento in `taxonomy/all_reference_species.json`, ma il training usa solo
le specie abbinate in modo non ambiguo e con almeno `MIN_OBS_PER_SPECIES` osservazioni utilizzabili.
I sinonimi sono confrontati contro l'intera checklist nazionale per evitare di attribuire una foto siciliana
per errore quando lo stesso nome è usato per più specie. Niente fuzzy matching automatico.
Un `no_usable_match` può dipendere da assenza nel CSV, sinonimia non risolta, ID/URL mancanti: non prova da solo
l'assenza della specie nell'intera collezione PlantCLEF. Il numero esatto coperto viene misurato sul TUO file.

## Colab gratuito e scala del problema
Migliaia di classi richiedono molti più dati e calcolo rispetto a 40. Il default usa al massimo 60 osservazioni
per specie come **prima prova estesa**, 30 estrazioni per specie/epoca e una vista per estrazione: non è una
configurazione finale per ottenere alta accuratezza su tutta la flora. La limitazione è dichiarata nei report.
Aumenta `MAX_OBS_PER_SPECIES` (o usa `None`) e `OBS_DRAWS_PER_SPECIES` in un **nuovo RUN_ID** quando hai risorse.
Le etichette delle classi rimangono tutte quelle allenabili: il limite riguarda le osservazioni, non il numero di specie.

La pipeline conserva split per osservazione, augmentation moderata, BatchNorm congelate, checkpoint con optimizer
su Drive, teacher opzionale e export TFLite. Le predizioni e la cache del teacher sono gestite a blocchi/disco;
una cache teacher con N immagini e C classi richiede circa **4×N×C byte locali**, più i blocchi compressi su Drive.
Non c'è garanzia che lo spazio Drive gratuito o la GPU assegnata bastino. Una rete MobileNetV3Small potrebbe
non avere capacità sufficiente per distinguere molte specie simili; ampliare le classi non garantisce qualità.

**Fonti:** [FlorItaly](https://dryades.units.it/floritaly/),
[snapshot e sinonimi](https://github.com/gibedini/floritaly/tree/6c4c7b08dc167a943907b2249b0a6b9684096f1d),
[PlantCLEF 2024](https://www.imageclef.org/PlantCLEF2024).


In [ ]:
# 1 — Usa TensorFlow già fornito da Colab, senza reinstallare lo stack CUDA.
# La cella controlla le API necessarie e stampa le versioni per la riproducibilità.
import os, sys, json, re, math, time, uuid, hashlib, shutil, tempfile, zipfile, gc, io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
assert tuple(map(int, tf.__version__.split('.')[:2])) >= (2, 16), 'Serve TensorFlow >= 2.16 / Keras 3.'
print('Python:', sys.version.split()[0], '| TensorFlow:', tf.__version__, '| Keras:', tf.keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('GPU:', gpus or 'ASSENTE: assegna una GPU prima del training; preparazione dati possibile su CPU.')


In [ ]:
# 2 — Le sole impostazioni da modificare prima del primo training.
# MODE può cambiare fra sessioni. Modifiche a dati/iperparametri richiedono un nuovo RUN_ID.
METADATA_CSV = Path('/content/drive/MyDrive/PlantCLEF2024singleplanttrainingdata.csv')
PROJECT = Path('/content/drive/MyDrive/Flora_Sicilia_v3')
RUN_ID = 'sicilia_v3_01'
MODE = 'baseline'                     # 'baseline', 'teacher', 'distill'
MAX_SESSION_MINUTES = 150             # budget prudente del loop, NON estende la sessione Colab
CHECKPOINT_MINUTES = 5                # più salvataggi = meno lavoro perso, più I/O su Drive
ALLOW_CPU_TRAINING = False
RUN_FINAL_TEST = False                # True SOLO dopo aver fissato il modello su validation
EXPORT_TFLITE = False                 # True quando vuoi esportare il modello selezionato
EXPORT_INT8 = False                   # opzionale: verifica l'eventuale perdita di accuratezza

SEED = 42
MAX_OBS_PER_SPECIES = 60              # prima prova estesa; None = tutte, con NUOVO RUN_ID
VAL_FRACTION, TEST_FRACTION = 0.15, 0.15
MIN_OBS_PER_SPECIES = 20               # blocca split troppo piccoli, non elimina specie in silenzio
IMG_SIZE, CACHE_MAX_SIDE, BATCH = 224, 384, 32
OBS_DRAWS_PER_SPECIES = 30             # prima prova estesa; aumentare in un NUOVO RUN_ID
VIEWS_PER_DRAW = 1                    # default economico; 2 se hai molte osservazioni multi-foto
DOWNLOAD_WORKERS, SHARD_SIZE = 6, 256
MAX_DOWNLOAD_FAILURE_FRACTION = 0.02  # oltre il 2%: riprovare prima del training
STUDENT_HEAD_EPOCHS, STUDENT_FT_EPOCHS = 4, 24
TEACHER_HEAD_EPOCHS, TEACHER_FT_EPOCHS = 3, 12
KD_EPOCHS = 12
PATIENCE, MIN_DELTA = 6, 0.001
KD_TEMPERATURE, KD_WEIGHT = 3.0, 0.4   # 60% etichette reali, 40% distillazione
TEACHER_MIN_GAIN = 0.005              # almeno +0,5 punti percentuali di macro-F1 su validation
# Float32 per semplificare compatibilità, ripresa ed export. BATCH=16 PRIMA di iniziare se GPU piccola.
# Non cambiare BATCH durante una ripresa: cambia il significato del cursore del checkpoint.
tf.keras.mixed_precision.set_global_policy('float32')
tf.keras.utils.set_random_seed(SEED)
assert MODE in {'baseline', 'teacher', 'distill'}
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
RUN = PROJECT / RUN_ID
LOCAL = Path('/content/flora_v2') / RUN_ID
for p in [RUN, LOCAL, RUN/'data', RUN/'reports', RUN/'stages', LOCAL/'images']:
    p.mkdir(parents=True, exist_ok=True)
print('Cartella persistente:', RUN)

# Prima produci il report di copertura, poi abilita il training del sottoinsieme documentato.
TRAINING_ENABLED = False
REQUIRE_FULL_REFERENCE_COVERAGE = False
INCLUDE_CASUAL_ALIENS = True
CLASS_SCOPE = 'sicilia_plus_dataset'  # include tutte le specie utilizzabili PlantCLEF; 'spontaneous_only' = checklist SIC


In [ ]:
"""FlorItaly 2023 -> species-level Sicily targets -> PlantCLEF inventory.

Reference: Gianni Bedini, Matteo Conti / floritaly; Checklist 27 April 2023.
Snapshot is versioned, not claimed to be a current or exhaustive 2026 checklist.
No fuzzy name matching. Subspecies/varieties are deliberately grouped at species level.
"""
import re
import hashlib
import unicodedata
from collections import defaultdict
import pandas as pd

REFERENCE_COMMIT = '6c4c7b08dc167a943907b2249b0a6b9684096f1d'
REFERENCE_URL = ('https://raw.githubusercontent.com/gibedini/floritaly/' + REFERENCE_COMMIT
                 + '/data-raw/CKL_27_04_2023.xlsx')
REFERENCE_SHA256 = '373c74e489b55868796151a8da1cc6a752c8bdb08ec6a0ed167a14ccec44feb8'
REFERENCE_DATE = '2023-04-27'


def species_name(value):
    """Conservative binomial parser; preserve named hybrids, reject formulas and qualifiers."""
    if value is None or pd.isna(value):
        return None
    text = ''.join(c for c in unicodedata.normalize('NFKD',str(value)) if not unicodedata.combining(c))
    text = re.sub(r'\s+', ' ', text.strip())
    if re.search(r'\?|\b(?:auct|sensu|cf|aff|non|nec)\b',text,re.I) or re.search(r'(?<!\w)p\.\s*p\.',text):
        return None
    if ' × ' in text and not re.match(r'^(?:×\s*)?[A-Z][a-z-]+ × [a-z-]+(?:\s|$)',text):
        return None
    match = re.match(r'^(×\s*)?([A-Z][a-z-]+)\s+(×\s*)?([a-z][a-z-]+)(?=\s|$)', text)
    if not match:
        return None
    genus_hybrid,genus,hybrid,epithet=match.groups()
    if epithet in {'sp','spp','sect','subsp','var','aggregate','agg'}:
        return None
    return ('×' if genus_hybrid else '') + genus + ' ' + ('×' if hybrid else '') + epithet


def norm(value):
    return (species_name(value) or '').casefold()


def build_reference(xlsx, include_casual=True, include_nonregional_names=False):
    ckl=pd.read_excel(xlsx,sheet_name='Checklist',keep_default_na=False)
    syn=pd.read_excel(xlsx,sheet_name='Sinonimi',keep_default_na=False)
    required={'Codice unico','entità','SIC'}
    if not required.issubset(ckl):
        raise ValueError('Schema checklist inatteso: verificare il formato.')
    ckl['canonical_species']=ckl['entità'].map(species_name)
    # (P) records at parent rank are grouped with their recorded subspecies.
    tokens=ckl.SIC.astype(str).str.replace(r'[()]',' ',regex=True).str.split()
    present=tokens.map(lambda t:'P' in t)
    if not include_casual:
        present &= ~tokens.map(lambda t:'CAS' in t)
    regional=ckl[present].copy()
    unparsed=regional[regional.canonical_species.isna()].copy()
    regional=regional.dropna(subset=['canonical_species'])
    targets=sorted(regional.canonical_species.unique())
    # Ambiguity is checked against the ENTIRE national checklist, including non-Sicilian taxa.
    destinations=defaultdict(set)
    accepted_by_id={str(row['Codice unico']):row['canonical_species']
                    for _,row in ckl.iterrows() if pd.notna(row['canonical_species'])}
    for name in ckl.canonical_species.dropna():
        destinations[norm(name)].add(name)
    for _,row in syn.iterrows():
        accepted=accepted_by_id.get(str(row['codice unico']))
        alias=str(row['sinonimo']).strip()
        # Known incorrect entry excluded by the source R package itself.
        if alias=='Sesleria apennina Ujhelyi':
            continue
        key=norm(alias)
        if accepted and key:
            destinations[key].add(accepted)
    target_set=set(targets)
    lookup={key:next(iter(values)) for key,values in destinations.items()
            if len(values)==1 and (include_nonregional_names or next(iter(values)) in target_set)}
    ambiguous=[{'normalized_name':key,'accepted_candidates':' | '.join(sorted(values))}
               for key,values in destinations.items() if len(values)>1 and (include_nonregional_names or values & target_set)]
    summary={'source_url':REFERENCE_URL,'snapshot_date':REFERENCE_DATE,
             'reference_sha256':REFERENCE_SHA256,'include_casual':include_casual,
             'selected_regional_taxa':int(present.sum()),'species_level_targets':len(targets),
             'unparsed_regional_taxa':len(unparsed),'ambiguous_names':len(ambiguous),
             'name_matching_scope':'national' if include_nonregional_names else 'regional',
             'scope':'vascular plants recorded as present in SIC; not all cultivated/ornamental species'}
    return targets,lookup,regional,unparsed,pd.DataFrame(ambiguous),summary


def audit_metadata(csv_path,targets,lookup,min_observations=20,chunksize=100000):
    """Exact counts of usable image IDs and independent observations, no image download."""
    observations=defaultdict(set)
    images=defaultdict(set)
    organ_counts=defaultdict(int)
    unmapped=defaultdict(int)
    invalid=defaultdict(int)
    inventory=defaultdict(int)
    per_class_source_names=defaultdict(set)
    columns=['species','obs_id','image_name','organ','image_backup_url','url']
    for chunk in pd.read_csv(csv_path,sep=';',usecols=columns,dtype=str,
                             keep_default_na=False,chunksize=chunksize):
        chunk=chunk.apply(lambda col:col.str.strip())
        for name,count in chunk.species.value_counts().items():
            inventory[name]+=int(count)
        mapped=chunk.species.map(norm).map(lookup)
        for name,count in chunk.loc[mapped.isna(),'species'].value_counts().items():
            unmapped[name]+=int(count)
        valid_url=chunk.image_backup_url.str.match(r'^https?://') | chunk.url.str.match(r'^https?://')
        valid=(chunk.obs_id!='') & (chunk.image_name!='') & valid_url
        for name,count in mapped[~valid].dropna().value_counts().items():
            invalid[name]+=int(count)
        chunk['canonical']=mapped
        for species,part in chunk[valid & mapped.notna()].groupby('canonical'):
            observations[species].update(part.obs_id)
            images[species].update(part.image_name)
            per_class_source_names[species].update(part.species)
            for organ,count in part.organ.str.casefold().replace('','unknown').value_counts().items():
                organ_counts[(species,organ)]+=int(count)
    records=[]
    for species in targets:
        obs_count=len(observations[species])
        status=('trainable' if obs_count>=min_observations else 'insufficient_observations'
                if obs_count else 'no_usable_match')
        records.append({'species':species,'observations':obs_count,'images':len(images[species]),
                        'invalid_rows':invalid[species],'status':status,
                        'plantclef_names':' | '.join(sorted(per_class_source_names[species]))})
    audit=pd.DataFrame(records)
    inventory_frame=pd.DataFrame([{'plantclef_name':k,'image_rows':v} for k,v in sorted(inventory.items())])
    unmatched=pd.DataFrame([{'plantclef_name':k,'image_rows':v} for k,v in sorted(unmapped.items())])
    organs=pd.DataFrame([{'species':sp,'organ':organ,'image_rows':count}
                        for (sp,organ),count in sorted(organ_counts.items())])
    return audit,inventory_frame,unmatched,organs


def extend_with_dataset_species(csv_path, regional_targets, national_lookup, ambiguous_names, chunksize=100000):
    """Include dataset species without claiming they occur (wild or cultivated) in Sicily."""
    names=set()
    for chunk in pd.read_csv(csv_path,sep=';',usecols=['species'],dtype=str,
                             keep_default_na=False,chunksize=chunksize):
        names.update(chunk.species.str.strip().unique())
    ambiguous=set(ambiguous_names.get('normalized_name',pd.Series(dtype=str)))
    lookup=dict(national_lookup)
    targets=set(regional_targets)
    unresolved=[]
    for name in sorted(names):
        key=norm(name)
        if not key or key in ambiguous:
            unresolved.append({'plantclef_name':name,'reason':'unparsed' if not key else 'ambiguous_taxonomy'})
            continue
        accepted=lookup.get(key)
        if accepted is None:
            accepted=species_name(name)
            lookup[key]=accepted
        targets.add(accepted)
    # Keep only names that can actually map to a chosen target.
    lookup={key:sp for key,sp in lookup.items() if sp in targets}
    return sorted(targets),lookup,pd.DataFrame(unresolved,columns=['plantclef_name','reason'])


# 3 — Costruisce le classi siciliane realmente supportate dal tuo CSV.
# L'XLSX è una sorgente versionata; non occorre scegliere manualmente migliaia di nomi.
import importlib.util, subprocess
if importlib.util.find_spec('openpyxl') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','openpyxl>=3.1,<4'])
TAXONOMY = RUN/'taxonomy'
TAXONOMY.mkdir(exist_ok=True)
reference_file = TAXONOMY/'FlorItaly_2023-04-27.xlsx'
if not reference_file.exists():
    response = requests.get(REFERENCE_URL,timeout=120)
    response.raise_for_status()
    assert hashlib.sha256(response.content).hexdigest()==REFERENCE_SHA256, 'Checksum FlorItaly errato.'
    temporary=reference_file.with_suffix('.partial')
    temporary.write_bytes(response.content)
    os.replace(temporary,reference_file)
assert hashlib.sha256(reference_file.read_bytes()).hexdigest()==REFERENCE_SHA256
assert CLASS_SCOPE in {'sicilia_plus_dataset','spontaneous_only'}
REGIONAL_TARGETS, full_lookup, regional, unparsed, ambiguous, reference_summary = build_reference(
    reference_file,include_casual=INCLUDE_CASUAL_ALIENS,include_nonregional_names=CLASS_SCOPE=='sicilia_plus_dataset')
ALL_TARGETS=REGIONAL_TARGETS
config=dict(reference_sha256=REFERENCE_SHA256,include_casual=INCLUDE_CASUAL_ALIENS,
            min_observations=MIN_OBS_PER_SPECIES,class_scope=CLASS_SCOPE,algorithm_version=2)
config_path=TAXONOMY/'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text())==config, 'Tassonomia cambiata: serve un nuovo RUN_ID.'
else:
    config_path.write_text(json.dumps(config,sort_keys=True))
coverage_file=TAXONOMY/'coverage.csv'
if coverage_file.exists() and (TAXONOMY/'audit_complete.json').exists():
    audit=pd.read_csv(coverage_file,keep_default_na=False)
    print('Audit recuperato da Drive. Per cambiare CSV o tassonomia usa un NUOVO RUN_ID.')
    saved=json.loads((TAXONOMY/'resolved_scope.json').read_text())
    ALL_TARGETS,full_lookup=saved['targets'],saved['lookup']
else:
    if not METADATA_CSV.exists():
        raise FileNotFoundError(f'CSV non trovato: {METADATA_CSV}')
    if CLASS_SCOPE=='sicilia_plus_dataset':
        ALL_TARGETS,full_lookup,unresolved=extend_with_dataset_species(METADATA_CSV,REGIONAL_TARGETS,full_lookup,ambiguous)
        unresolved.to_csv(TAXONOMY/'dataset_unresolved_taxonomy.csv',index=False)
    (TAXONOMY/'resolved_scope.json').write_text(json.dumps({'targets':ALL_TARGETS,'lookup':full_lookup},ensure_ascii=False))
    audit,inventory,unmatched,organs=audit_metadata(METADATA_CSV,ALL_TARGETS,full_lookup,MIN_OBS_PER_SPECIES)
    audit['sicilia_reference_2023']=audit.species.isin(REGIONAL_TARGETS)
    audit['regional_evidence']=np.where(audit.sicilia_reference_2023,'present_in_2023_checklist','additional_dataset_species_region_not_verified')
    inventory.to_csv(TAXONOMY/'plantclef_inventory.csv',index=False)
    unmatched.to_csv(TAXONOMY/'plantclef_unmatched_names.csv',index=False)
    organs.to_csv(TAXONOMY/'matched_organ_coverage.csv',index=False)
    audit.to_csv(coverage_file,index=False)
    h=hashlib.sha256()
    with open(METADATA_CSV,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    (TAXONOMY/'audit_complete.json').write_text(json.dumps({'csv_sha256':h.hexdigest(),'csv_name':METADATA_CSV.name}))
regional[['Codice unico','entità','SIC','canonical_species']].to_csv(TAXONOMY/'reference_taxa.csv',index=False)
unparsed.to_csv(TAXONOMY/'non_species_or_unparsed.csv',index=False)
ambiguous.to_csv(TAXONOMY/'ambiguous_names.csv',index=False)
(TAXONOMY/'reference_source.json').write_text(json.dumps(reference_summary,ensure_ascii=False,indent=2))
(TAXONOMY/'all_reference_species.json').write_text(json.dumps(REGIONAL_TARGETS,ensure_ascii=False,indent=2))
TARGET_SPECIES=sorted(audit.loc[audit.status=='trainable','species'])
CLASSES=TARGET_SPECIES
N_CLASSES=len(CLASSES)
assert N_CLASSES>=2,'Non ci sono abbastanza classi utilizzabili: controlla i report.'
CLASS_TO_ID={sp:i for i,sp in enumerate(CLASSES)}
lookup={key:sp for key,sp in full_lookup.items() if sp in CLASS_TO_ID}
ALIASES={sp:sorted(key for key,value in lookup.items() if value==sp) for sp in CLASSES}
(TAXONOMY/'trainable_species.json').write_text(json.dumps(CLASSES,ensure_ascii=False,indent=2))
print('Fonte:',REFERENCE_DATE,'| specie del riferimento SIC:',len(REGIONAL_TARGETS),'| classi allenabili:',N_CLASSES)
display(audit.groupby('status').agg(species=('species','size'),images=('images','sum'),observations=('observations','sum')))
display(audit[audit.status!='trainable'].head(30))
print('Elenco COMPLETO delle lacune:',coverage_file)
print('Le specie mancanti o con pochi dati NON sono riconosciute dal modello.')
planned_obs=audit.loc[audit.status=='trainable','observations']
if MAX_OBS_PER_SPECIES is not None:planned_obs=planned_obs.clip(upper=MAX_OBS_PER_SPECIES)
print('Osservazioni selezionabili prima dello split:',int(planned_obs.sum()))
print('Esempi/batch indicativi per epoca:',N_CLASSES*(OBS_DRAWS_PER_SPECIES or 30)*VIEWS_PER_DRAW,
      '/',math.ceil(N_CLASSES*(OBS_DRAWS_PER_SPECIES or 30)*VIEWS_PER_DRAW/BATCH))
print('Specie aggiuntive: presenza siciliana NON verificata; la lista non è un censimento delle ornamentali locali.')
if REQUIRE_FULL_REFERENCE_COVERAGE and ((audit.status!='trainable') & audit.species.isin(REGIONAL_TARGETS)).any():
    raise RuntimeError('Copertura completa richiesta ma non disponibile: integra dati/alias verificati prima di allenare.')
if not TRAINING_ENABLED:
    raise RuntimeError('AUDIT COMPLETATO. Leggi taxonomy/coverage.csv. Per allenare le classi disponibili imposta TRAINING_ENABLED=True nella cella 2 e riesegui.')


## Dati persistenti e controlli contro contaminazione
Il CSV è letto a blocchi. Si separano **osservazioni**, non singole foto; validation e test conservano
la distribuzione naturale delle specie. Gli ID mancanti e le etichette in conflitto bloccano la preparazione.
Si salva la selezione prima dei download: una nuova sessione non rifà lo split.

Le immagini vengono corrette per orientamento EXIF, convertite in RGB e ridotte a massimo 384 pixel sul lato lungo.
Gli archivi da 256 immagini stanno su Drive, le copie di lavoro nel disco veloce `/content`.
Il ridimensionamento del modello usa padding, evitando lo stiramento dell'immagine.
I download incompleti si possono riprendere; dopo il primo manifest definitivo, il dataset resta congelato.
Duplicati esatti dopo normalizzazione vengono rimossi, inclusi quelli tra split; duplicati visivi quasi identici
richiedono un controllo ulteriore e non sono rilevati automaticamente.


In [ ]:
# 4 — Scritture controllate e firme: un checkpoint non deve essere riusato con classi o dati differenti.
def digest_bytes(data):
    return hashlib.sha256(data).hexdigest()

def file_digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def canonical(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()

def atomic_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.partial-' + uuid.uuid4().hex)
    try:
        tmp.write_bytes(data)
        assert file_digest(tmp) == digest_bytes(data), 'Scrittura incompleta: ' + str(path)
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)

def atomic_json(path, obj):
    atomic_bytes(path, canonical(obj))

def atomic_csv(path, df):
    atomic_bytes(path, df.to_csv(index=False).encode())

def lock_json(path, obj):
    if Path(path).exists():
        if json.loads(Path(path).read_text()) != obj:
            raise ValueError(f'Configurazione diversa da {path}. Ripristinala o usa un nuovo RUN_ID.')
    else:
        atomic_json(path, obj)

DATA_CONFIG = dict(seed=SEED, classes=CLASSES, aliases=ALIASES, cap=MAX_OBS_PER_SPECIES,
                   val=VAL_FRACTION, test=TEST_FRACTION, min_obs=MIN_OBS_PER_SPECIES,
                   cache_side=CACHE_MAX_SIDE, shard_size=SHARD_SIZE, pipeline_version=2)
lock_json(RUN/'data_config.json', DATA_CONFIG)
lock_json(RUN/'classes.json', CLASSES)


In [ ]:
# 5 — Estrae le 40 specie e crea lo split per osservazione, una sola volta.
def make_selection(csv_path):
    required = ['image_name', 'organ', 'obs_id', 'species', 'image_backup_url', 'url']
    parts = []
    for chunk in pd.read_csv(csv_path, sep=';', usecols=required, dtype=str,
                             keep_default_na=False, chunksize=100_000):
        names = chunk.species.map(norm)
        keep = chunk[names.isin(lookup)].copy()
        keep['flora_species'] = names[names.isin(lookup)].map(lookup)
        parts.append(keep)
    if not parts:
        raise ValueError('CSV senza righe.')
    df = pd.concat(parts, ignore_index=True)
    for col in required:
        df[col] = df[col].str.strip()
    if df.empty:
        raise ValueError('Nessuna delle specie richieste trovata nel CSV.')
    if (df.obs_id == '').any() or (df.image_name == '').any():
        raise ValueError('obs_id o image_name mancanti: correggere i metadati prima dello split.')
    if (df.groupby('obs_id').flora_species.nunique() > 1).any():
        raise ValueError('Una stessa osservazione ha più specie: serve verificare le etichette.')
    if (df.groupby('image_name').obs_id.nunique() > 1).any():
        raise ValueError('Uno stesso image_name appartiene a osservazioni diverse: verificare gli ID.')
    df = df.drop_duplicates(['image_name']).copy()
    df['organ'] = df.organ.str.casefold().replace('', 'unknown')
    obs = df[['flora_species', 'obs_id']].drop_duplicates()
    counts = obs.groupby('flora_species').size().reindex(CLASSES, fill_value=0)
    display(counts.rename('osservazioni disponibili').to_frame())
    if (counts < MIN_OBS_PER_SPECIES).any():
        raise ValueError('Specie assenti o troppo rare: controlla il CSV e i sinonimi. Non vengono escluse automaticamente.')
    selected = []
    rng = np.random.default_rng(SEED)
    for sp in CLASSES:
        g = obs[obs.flora_species == sp].sort_values('obs_id').copy()
        g = g.iloc[rng.permutation(len(g))].reset_index(drop=True)
        if MAX_OBS_PER_SPECIES is not None:
            g = g.iloc[:MAX_OBS_PER_SPECIES].copy()
        nv, nt = max(1, int(len(g)*VAL_FRACTION)), max(1, int(len(g)*TEST_FRACTION))
        if len(g)-nv-nt < 2:
            raise ValueError('Troppo poche osservazioni per split affidabili.')
        g['split'] = ['val']*nv + ['test']*nt + ['train']*(len(g)-nv-nt)
        selected.append(g)
    df = df.merge(pd.concat(selected), on=['flora_species', 'obs_id'], validate='many_to_one')
    df['image_id'] = [digest_bytes(canonical([s, o, n]))[:32]
                      for s, o, n in zip(df.flora_species, df.obs_id, df.image_name)]
    df['label'] = df.flora_species.map(CLASS_TO_ID).astype(int)
    df = df.sort_values('image_id').reset_index(drop=True)
    df['shard'] = np.arange(len(df)) // SHARD_SIZE
    assert df.groupby('obs_id').split.nunique().max() == 1
    return df

selection_file = RUN/'data'/'selection.csv'
if selection_file.exists():
    selected = pd.read_csv(selection_file, dtype={'obs_id':str}, keep_default_na=False)
    print('Split recuperato da Drive:', len(selected), 'immagini.')
else:
    if not METADATA_CSV.exists():
        raise FileNotFoundError(f'Carica il CSV in {METADATA_CSV} oppure correggi METADATA_CSV.')
    selected = make_selection(METADATA_CSV)
    atomic_csv(selection_file, selected)
    atomic_json(RUN/'data'/'source.json', {'csv_name':METADATA_CSV.name, 'sha256':file_digest(METADATA_CSV)})
display(selected.groupby(['flora_species','split']).obs_id.nunique().unstack(fill_value=0))


In [ ]:
# 6 — Download riprendibile, timeout/retry, verifica immagini, archivi persistenti.
# Se un archivio è interrotto/corrotto, non viene considerato completo.
import threading
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
_thread_local = threading.local()

def download_image(row):
    session = getattr(_thread_local, 'session', None)
    if session is None:
        session = requests.Session()
        retry = Retry(total=2, backoff_factor=0.5, status_forcelist=[429,500,502,503,504])
        session.mount('https://', HTTPAdapter(max_retries=retry))
        session.mount('http://', HTTPAdapter(max_retries=retry))
        _thread_local.session = session
    error = 'URL assente'
    for url in dict.fromkeys([row['image_backup_url'], row['url']]):
        if not isinstance(url, str) or not url.startswith(('https://','http://')):
            continue
        try:
            with session.get(url, timeout=(10,30), stream=True) as response:
                response.raise_for_status()
                payload = bytearray()
                for chunk in response.iter_content(65536):
                    payload.extend(chunk)
                    if len(payload) > 30*1024*1024:
                        raise ValueError('Immagine oltre 30 MB')
            with Image.open(io.BytesIO(payload)) as im:
                im = ImageOps.exif_transpose(im).convert('RGB')
                im.thumbnail((CACHE_MAX_SIDE,CACHE_MAX_SIDE), Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                im.save(buf, 'JPEG', quality=92)
            return row['image_id'], buf.getvalue(), None
        except Exception as e:
            # Non stampare URL completi: potrebbero contenere parametri di accesso.
            error = type(e).__name__
    return row['image_id'], None, error

def copy_verified(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial-' + uuid.uuid4().hex)
    try:
        shutil.copyfile(src, tmp)
        assert file_digest(src) == file_digest(tmp), 'Copia incompleta: ' + str(dst)
        os.replace(tmp, dst)
    finally:
        tmp.unlink(missing_ok=True)

def prepare_images(selection, frozen=None):
    metadata, failed = [], []
    required_ids = set(frozen.image_id) if frozen is not None else None
    for shard, group in tqdm(selection.groupby('shard'), desc='Archivi immagini'):
        archive = RUN/'data'/f'images-{int(shard):05d}.zip'
        blobs = {}
        if archive.exists():
            try:
                with zipfile.ZipFile(archive) as z:
                    assert z.testzip() is None
                    # Legge solo nomi attesi; nessuna estrazione di percorsi arbitrari.
                    for image_id in group.image_id:
                        name = image_id + '.jpg'
                        if name in z.namelist():
                            blobs[image_id] = z.read(name)
            except (zipfile.BadZipFile, AssertionError, OSError):
                if frozen is not None:
                    raise RuntimeError(f'Archivio non valido {archive}: ripristinalo da Drive prima di riprendere.')
                print('Ricostruzione archivio incompleto:', archive.name)
                blobs = {}
        target = group if required_ids is None else group[group.image_id.isin(required_ids)]
        missing = target[~target.image_id.isin(blobs)]
        if frozen is not None and len(missing):
            raise RuntimeError('Il dataset è congelato ma un archivio manca: ripristina il backup, non cambiare immagini a metà training.')
        if len(missing):
            with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
                for image_id, blob, error in pool.map(download_image, missing.to_dict('records')):
                    if blob is not None:
                        blobs[image_id] = blob
                    else:
                        failed.append({'image_id':image_id, 'error':error})
            local_zip = LOCAL/f'download-{int(shard):05d}.zip'
            with zipfile.ZipFile(local_zip, 'w', compression=zipfile.ZIP_STORED) as z:
                for image_id, blob in blobs.items():
                    z.writestr(image_id+'.jpg', blob)
            copy_verified(local_zip, archive)
            local_zip.unlink()
        for image_id in target.image_id:
            if image_id in blobs:
                blob = blobs[image_id]
                p = LOCAL/'images'/(image_id+'.jpg')
                sha = digest_bytes(blob)
                if not p.exists() or file_digest(p) != sha:
                    atomic_bytes(p, blob)
                metadata.append({'image_id':image_id, 'sha256':sha})
    if failed:
        atomic_csv(RUN/'reports'/'download_failures.csv', pd.DataFrame(failed))
    return pd.DataFrame(metadata), failed

manifest_file = RUN/'data'/'manifest.csv'
frozen = pd.read_csv(manifest_file, dtype={'obs_id':str}, keep_default_na=False) if manifest_file.exists() else None
image_meta, failures = prepare_images(selected, frozen)
if frozen is None:
    failure_rate = 1 - len(image_meta)/len(selected)
    print(f'Immagini mancanti: {len(selected)-len(image_meta)} / {len(selected)} ({failure_rate:.1%})')
    if failure_rate > MAX_DOWNLOAD_FAILURE_FRACTION:
        raise RuntimeError('Troppi download falliti. Riesegui questa cella: recupera le immagini mancanti, senza riscaricare le altre.')
    manifest = selected.merge(image_meta, on='image_id', validate='one_to_one')
    # Scarta TUTTE le copie di un contenuto presente in più split o con etichette in conflitto.
    grouped = manifest.groupby('sha256')
    bad = set(grouped.split.nunique().loc[lambda s:s>1].index) | set(grouped.label.nunique().loc[lambda s:s>1].index)
    duplicates = manifest[manifest.sha256.isin(bad) | manifest.duplicated('sha256', keep='first')]
    atomic_csv(RUN/'reports'/'excluded_duplicates.csv', duplicates)
    manifest = manifest[~manifest.sha256.isin(bad)].drop_duplicates('sha256').reset_index(drop=True)
    print('Copie escluse:', len(duplicates))
else:
    manifest = frozen
    expected = dict(zip(frozen.image_id, frozen.sha256))
    assert dict(zip(image_meta.image_id, image_meta.sha256)) == expected, 'Il contenuto delle immagini è cambiato.'

counts = manifest.groupby(['flora_species','split']).obs_id.nunique().unstack(fill_value=0).reindex(CLASSES, fill_value=0)
assert all(s in counts.columns for s in ['train','val','test']), 'Uno split è vuoto.'
assert (counts[['train','val','test']] >= 3).all().all(), 'Servono almeno 3 osservazioni per specie in ogni split dopo download/deduplica.'
assert manifest.groupby('obs_id').split.nunique().max() == 1
assert manifest.groupby('sha256').split.nunique().max() == 1
if frozen is None:
    atomic_csv(manifest_file, manifest)
DATA_HASH = file_digest(manifest_file)
manifest['path'] = manifest.image_id.map(lambda x: str(LOCAL/'images'/(x+'.jpg')))
frames = {s:manifest[manifest.split==s].reset_index(drop=True) for s in ['train','val','test']}
atomic_csv(RUN/'reports'/'observation_counts.csv', counts.reset_index())
organ_report = manifest.groupby(['flora_species','split','organ']).size().rename('images').reset_index()
atomic_csv(RUN/'reports'/'organ_coverage.csv', organ_report)
display(counts)
display(organ_report[organ_report.split=='train'].pivot(index='flora_species',columns='organ',values='images').fillna(0).astype(int))
print('Dataset congelato:', DATA_HASH[:12], '| immagini:', len(manifest))


## Campionamento e augmentation
Ogni epoca estrae lo stesso numero di osservazioni per specie (default: mediana delle numerosità train).
Le osservazioni vengono percorse in ordine casuale, ripetendo solo quando necessario. Per ogni estrazione si
alternano gli organi disponibili tra epoche. Default: una vista per estrazione, perché il CSV originale ha
appena 1,06 immagini/osservazione. Con `VIEWS_PER_DRAW=2` si preferiscono organi e foto diversi nella stessa
estrazione; se esiste una sola foto, ripeterla non crea un'osservazione nuova e raddoppia il lavoro.
L'epoca bilanciata non è un passaggio esaustivo su tutte le immagini.

Le trasformazioni sono moderate: flip orizzontale, crop fino al 10%, luminosità e contrasto contenuti.
Il padding conserva le proporzioni. MobileNetV3 ed EfficientNet includono già il preprocessing e ricevono
pixel **float32 in [0,255]**: non dividere nuovamente per 255.

Per la distillazione si usa la stessa vista canonica per teacher e student (senza augmentation casuale):
una scelta esplicita per riusare logits offline senza associarli a crop diversi. La baseline iniziale ha già
imparato con augmentation; la distillazione è un affinamento opzionale da validare.


In [ ]:
# 7 — Ordine dei batch ricostruibile: epoca + indice del campione determinano scelta e augmentation.
train_frame = frames['train']
quota = OBS_DRAWS_PER_SPECIES or int(train_frame.groupby('label').obs_id.nunique().median())
assert quota > 0 and VIEWS_PER_DRAW > 0
STEPS_PER_EPOCH = math.ceil(quota*N_CLASSES*VIEWS_PER_DRAW/BATCH)
print('Estrazioni osservazioni/specie:', quota, '| batch/epoca:', STEPS_PER_EPOCH)

def epoch_indices(frame, epoch):
    rng = np.random.default_rng(SEED + int(epoch)*1009)
    indices = []
    for label, part in frame.groupby('label',sort=True):
        groups = {o:g for o,g in part.groupby('obs_id', sort=True)}
        obs_ids = np.array(sorted(groups))
        sequence = []
        while len(sequence) < quota:
            sequence.extend(rng.permutation(obs_ids).tolist())
        for obs_id in sequence[:quota]:
            g = groups[obs_id]
            organs = list(rng.permutation(sorted(g.organ.unique())))
            used = set()
            for v in range(VIEWS_PER_DRAW):
                candidates = g[g.organ==organs[v % len(organs)]]
                unused = candidates.loc[~candidates.index.isin(used)]
                if len(unused):
                    candidates = unused
                idx = int(rng.choice(candidates.index.to_numpy()))
                used.add(idx)
                indices.append(idx)
    return np.array(indices, dtype=np.int64)[rng.permutation(len(indices))]

def decode_image(path):
    im = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize_with_pad(tf.cast(im,tf.float32), IMG_SIZE, IMG_SIZE, antialias=True)

def augment_image(im, seed):
    seeds = tf.random.experimental.stateless_split(seed, 4)
    im = tf.image.stateless_random_flip_left_right(im, seeds[0])
    extra = max(1, int(IMG_SIZE*0.10))
    im = tf.image.resize(im, [IMG_SIZE+extra,IMG_SIZE+extra])
    im = tf.image.stateless_random_crop(im, [IMG_SIZE,IMG_SIZE,3], seeds[1])
    im = tf.image.stateless_random_brightness(im, max_delta=12.0, seed=seeds[2])
    im = tf.image.stateless_random_contrast(im, lower=0.85, upper=1.15, seed=seeds[3])
    return tf.clip_by_value(im, 0., 255.)

def training_dataset(epoch, start_batch=0, teacher_logits=None):
    order = epoch_indices(train_frame, epoch)
    # Taglia PRIMA del decoding, mantenendo i semi e il batch esatti dopo la ripresa.
    positions = np.arange(len(order), dtype=np.int32)
    offset = int(start_batch)*BATCH
    order, positions = order[offset:], positions[offset:]
    paths = train_frame.path.to_numpy()[order].astype(str)
    labels = train_frame.label.to_numpy(dtype=np.int32)[order]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, positions, order))
    def load(p,y,pos,row_index):
        x = decode_image(p)
        if teacher_logits is None:
            x = augment_image(x, tf.stack([tf.cast(SEED+int(epoch)*1009,tf.int32),pos]))
        if teacher_logits is None:
            target=tf.zeros([1],tf.float32)  # nessuna matrice N_immagini × N_classi per la baseline
        else:
            target=tf.numpy_function(lambda i:np.asarray(teacher_logits[int(i)],dtype=np.float32),[row_index],tf.float32)
            target.set_shape([N_CLASSES])
        return x,y,target
    opts = tf.data.Options()
    opts.experimental_deterministic = True
    return ds.with_options(opts).map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(2)

def evaluation_dataset(frame):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy().astype(str),frame.label.to_numpy(dtype=np.int32)))
    return ds.map(lambda p,y:(decode_image(p),y),num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(2)

def predict_frame(model, frame):
    # Solo per piccoli blocchi (cache teacher ed export); usare evaluate_frame per split interi.
    if len(frame)*N_CLASSES*4 > 256*1024*1024:
        raise MemoryError('Predizioni dense troppo grandi: usare evaluate_frame o blocchi più piccoli.')
    return np.concatenate([model(x,training=False).numpy() for x,_ in evaluation_dataset(frame)])

def evaluate_frame(model, frame):
    true,predicted,hits=[],[],[]
    for x,y in evaluation_dataset(frame):
        values=model(x,training=False).numpy()
        labels=y.numpy()
        true.extend(labels.tolist())
        predicted.extend(values.argmax(axis=1).tolist())
        top=np.argpartition(values,-min(3,N_CLASSES),axis=1)[:,-min(3,N_CLASSES):]
        hits.extend(np.any(top==labels[:,None],axis=1).tolist())
    return {'top1':float(accuracy_score(true,predicted)),'top3':float(np.mean(hits)),
            'macro_f1':float(f1_score(true,predicted,labels=list(range(N_CLASSES)),average='macro',zero_division=0))}

def image_metrics(frame, logits):
    y = frame.label.to_numpy(dtype=int)
    pred = np.argmax(logits, axis=1)
    return {'top1':float(accuracy_score(y,pred)),
            'top3':float(np.mean([a in b for a,b in zip(y,np.argsort(logits,axis=1)[:,-min(3,N_CLASSES):])])),
            'macro_f1':float(f1_score(y,pred,labels=list(range(N_CLASSES)),average='macro',zero_division=0))}


## Modelli e checkpoint di training
La testa si allena inizialmente con backbone congelato. Nel fine-tuning si sblocca l'ultimo 50% dei layer;
le BatchNormalization restano congelate per stabilità con batch piccoli. Learning rate con decadimento coseno,
label smoothing leggero e early stopping su macro-F1.

Un checkpoint è un archivio completo: pesi correnti, slot Adam, contatore dell'optimizer (quindi posizione del
learning rate), epoca, prossimo batch, best score, pazienza e pesi migliori. Il campionamento e l'augmentation
usano semi derivati dalla posizione; non ci sono dropout o RNG stateful da ricostruire.
I salvataggi vengono verificati prima di sostituire i precedenti; rimangono le ultime due copie per fase.
Se l'ultima copia è corrotta si prova la precedente. L'interruzione improvvisa può perdere al più il lavoro
successivo all'ultimo salvataggio completato; la sincronizzazione del mount Drive non è una garanzia assoluta.

Il limite di 150 minuti riguarda i loop di training della sessione, non download, validazione o conversione.
Alla scadenza si salva e la cella si interrompe intenzionalmente: alla sessione successiva riesegui dall'inizio.


In [ ]:
# 8 — Architetture standard Keras; output logits, softmax applicata solo dove serve.
def make_model(kind, weights='imagenet'):
    tf.keras.utils.set_random_seed(SEED)
    if kind == 'student':
        base = tf.keras.applications.MobileNetV3Small(
            input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, pooling='avg',
            include_preprocessing=True, weights=weights)
    elif kind == 'teacher':
        base = tf.keras.applications.EfficientNetB0(
            input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, pooling='avg', weights=weights)
    else:
        raise ValueError(kind)
    x = tf.keras.Input((IMG_SIZE,IMG_SIZE,3), name='rgb_0_255')
    # training=False mantiene BatchNorm e l'eventuale stochastic depth del teacher in modalità inference.
    features = base(x, training=False)
    logits = tf.keras.layers.Dense(N_CLASSES, dtype='float32', name='species_logits')(features)
    return tf.keras.Model(x,logits,name='flora_'+kind), base

def set_trainable(base, fine_tune):
    base.trainable = bool(fine_tune)
    if fine_tune:
        cut = int(len(base.layers)*0.5)
        for i,layer in enumerate(base.layers):
            layer.trainable = i >= cut and not isinstance(layer,tf.keras.layers.BatchNormalization)

TRAIN_CONFIG = dict(data_hash=DATA_HASH, classes=CLASSES, image_size=IMG_SIZE, batch=BATCH,
    seed=SEED, quota=quota, views=VIEWS_PER_DRAW, student_head=STUDENT_HEAD_EPOCHS,
    student_ft=STUDENT_FT_EPOCHS, teacher_head=TEACHER_HEAD_EPOCHS, teacher_ft=TEACHER_FT_EPOCHS,
    kd_epochs=KD_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, temperature=KD_TEMPERATURE,
    kd_weight=KD_WEIGHT, teacher_min_gain=TEACHER_MIN_GAIN, precision='float32',
    tensorflow=tf.__version__, keras=tf.keras.__version__, algorithm_version=3)
# Versioni diverse possono rendere incompatibili i checkpoint: fermati invece di ripartire in silenzio.
lock_json(RUN/'training_config.json', TRAIN_CONFIG)
TRAIN_HASH = digest_bytes(canonical(TRAIN_CONFIG))

class Progress(tf.Module):
    def __init__(self):
        super().__init__()
        self.epoch = tf.Variable(0,trainable=False,dtype=tf.int64)
        self.batch = tf.Variable(0,trainable=False,dtype=tf.int64)
        self.best = tf.Variable(-1.,trainable=False,dtype=tf.float32)
        self.wait = tf.Variable(0,trainable=False,dtype=tf.int64)
        self.done = tf.Variable(False,trainable=False,dtype=tf.bool)


In [ ]:
# 9 — Snapshot transazionali. Non usare questi archivi con altri notebook/configurazioni.
class SnapshotStore:
    def __init__(self, name, checkpoint, progress, local_dir):
        self.remote = RUN/'stages'/name
        self.remote.mkdir(parents=True,exist_ok=True)
        self.local = Path(local_dir)
        self.local.mkdir(parents=True,exist_ok=True)
        self.ckpt, self.progress = checkpoint, progress
        self.signature = digest_bytes(canonical({'training':TRAIN_HASH,'stage':name}))
        self.best_path = self.local/'best.weights.h5'

    def save(self):
        with tempfile.TemporaryDirectory(dir=LOCAL) as d:
            d = Path(d)
            self.ckpt.write(str(d/'state'))
            shutil.copyfile(self.best_path, d/'best.weights.h5')
            meta = {'signature':self.signature, 'epoch':int(self.progress.epoch.numpy()),
                    'batch':int(self.progress.batch.numpy()), 'files':{p.name:file_digest(p) for p in d.iterdir()}}
            (d/'metadata.json').write_bytes(canonical(meta))
            packed = self.local/'snapshot.zip'
            with zipfile.ZipFile(packed,'w',compression=zipfile.ZIP_STORED) as z:
                for p in d.iterdir():
                    z.write(p,p.name)
            name = f'snapshot-{time.time_ns():020d}-{uuid.uuid4().hex[:6]}.zip'
            copy_verified(packed, self.remote/name)
            packed.unlink()
        # Rimuove solo snapshot di questa fase, dopo la conferma della nuova copia.
        for old in sorted(self.remote.glob('snapshot-*.zip'))[:-2]:
            old.unlink()

    def restore(self):
        candidates = sorted(self.remote.glob('snapshot-*.zip'),reverse=True)
        for candidate in candidates:
            try:
                with zipfile.ZipFile(candidate) as z:
                    if z.testzip() is not None:
                        raise ValueError('CRC non valido')
                    meta = json.loads(z.read('metadata.json'))
                    if meta['signature'] != self.signature:
                        raise RuntimeError('Checkpoint incompatibile: usare il RUN_ID originale e la stessa configurazione.')
                    for name, sha in meta['files'].items():
                        if Path(name).name != name:
                            raise ValueError('Nome file non valido')
                        blob = z.read(name)
                        if digest_bytes(blob) != sha:
                            raise ValueError('Checksum non valido')
                        (self.local/name).write_bytes(blob)
            except (zipfile.BadZipFile,ValueError,KeyError,OSError) as e:
                print('Snapshot non leggibile, provo il precedente:', candidate.name, type(e).__name__)
                continue
            self.ckpt.read(str(self.local/'state')).assert_consumed()
            print('Ripresa:', self.remote.name, '| epoca',int(self.progress.epoch.numpy())+1,
                  '| prossimo batch',int(self.progress.batch.numpy())+1)
            return True
        if candidates:
            raise RuntimeError('Nessuno snapshot valido. Ripristina il backup; il training NON viene azzerato automaticamente.')
        return False


In [ ]:
# 10 — Loop condiviso da baseline, teacher e distillazione; un salvataggio anche a ogni fine epoca.
SESSION_START = time.monotonic()
SESSION_DEADLINE = SESSION_START + MAX_SESSION_MINUTES*60

class SessionPaused(RuntimeError):
    pass

def run_stage(model, base, name, fine_tune, epochs, initial_lr, teacher_logits=None):
    set_trainable(base, fine_tune)
    schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_lr, decay_steps=max(1,epochs*STEPS_PER_EPOCH), alpha=0.1)
    optimizer = tf.keras.optimizers.Adam(learning_rate=schedule, global_clipnorm=5.0)
    optimizer.build(model.trainable_variables)  # crea gli slot PRIMA del ripristino
    state = Progress()
    checkpoint = tf.train.Checkpoint(model=model,optimizer=optimizer,progress=state)
    store = SnapshotStore(name,checkpoint,state,LOCAL/'stages'/name)
    if not store.restore():
        model.save_weights(store.best_path)
        store.save()  # persiste anche il punto iniziale
    if bool(state.done.numpy()):
        model.load_weights(store.best_path)
        return model
    if not gpus and not ALLOW_CPU_TRAINING:
        raise RuntimeError('Training fermato: assegna una GPU Colab, poi riesegui. I dati sono già salvati.')

    @tf.function(reduce_retracing=True)
    def train_step(x,y,target):
        with tf.GradientTape() as tape:
            logits = model(x,training=True)
            onehot = tf.one_hot(y,N_CLASSES)
            hard = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(
                onehot,logits,from_logits=True,label_smoothing=0.05))
            loss = hard
            if teacher_logits is not None:
                temp = tf.cast(KD_TEMPERATURE,tf.float32)
                teacher_logp = tf.nn.log_softmax(target/temp,axis=-1)
                student_logp = tf.nn.log_softmax(logits/temp,axis=-1)
                kl = tf.reduce_mean(tf.reduce_sum(tf.exp(teacher_logp)*(teacher_logp-student_logp),axis=-1))
                loss = (1-KD_WEIGHT)*hard + KD_WEIGHT*temp*temp*kl
            tf.debugging.assert_all_finite(loss,'Loss non finita: controllare dati/learning rate.')
        gradients = tape.gradient(loss,model.trainable_variables)
        optimizer.apply_gradients(zip(gradients,model.trainable_variables))
        return loss

    last_save = time.monotonic()
    try:
        while int(state.epoch.numpy()) < epochs and int(state.wait.numpy()) < PATIENCE:
            epoch, start = int(state.epoch.numpy()), int(state.batch.numpy())
            bar = tqdm(training_dataset(epoch,start,teacher_logits), total=STEPS_PER_EPOCH-start,
                       desc=f'{name} epoca {epoch+1}/{epochs}')
            for x,y,target in bar:
                if time.monotonic() >= SESSION_DEADLINE:
                    store.save()
                    raise SessionPaused('Budget raggiunto: checkpoint salvato. Riprendi in una nuova sessione eseguendo dall’inizio.')
                loss = train_step(x,y,target)
                state.batch.assign_add(1)
                bar.set_postfix(loss=f'{float(loss.numpy()):.3f}')
                if time.monotonic()-last_save >= CHECKPOINT_MINUTES*60:
                    store.save()
                    last_save = time.monotonic()
            # Valuta tutte le immagini val, senza ricampionamento e senza augmentation.
            metrics = evaluate_frame(model,frames['val'])
            improved = metrics['macro_f1'] > float(state.best.numpy()) + MIN_DELTA
            if improved:
                state.best.assign(metrics['macro_f1'])
                state.wait.assign(0)
                model.save_weights(store.best_path)
            else:
                state.wait.assign_add(1)
            state.epoch.assign_add(1)
            state.batch.assign(0)
            row = dict(stage=name,epoch=int(state.epoch.numpy()),optimizer_steps=int(optimizer.iterations.numpy()),**metrics)
            atomic_json(RUN/'reports'/f'{name}-epoch-{int(state.epoch.numpy()):03d}.json',row)
            print(row)
            state.done.assign(int(state.epoch.numpy())>=epochs or int(state.wait.numpy())>=PATIENCE)
            store.save()
            last_save = time.monotonic()
    except KeyboardInterrupt:
        store.save()
        raise SessionPaused('Stop manuale: salvato il cursore dell’ultimo batch completato. Riesegui dall’inizio per riprendere.')
    state.done.assign(True)
    store.save()
    model.load_weights(store.best_path)
    return model

def save_model(model,path):
    tmp = LOCAL/('export-'+uuid.uuid4().hex+'.keras')
    model.save(tmp)
    copy_verified(tmp,path)
    tmp.unlink()

def train_pipeline(kind):
    final_file = RUN/(kind+'_best.keras')
    if final_file.exists() and (RUN/'reports'/(kind+'_validation.json')).exists():
        print('Percorso già completato, recupero:', final_file.name)
        return tf.keras.models.load_model(final_file,compile=False)
    has_resume = any((RUN/'stages'/(kind+'_head')).glob('snapshot-*.zip'))
    model,base = make_model(kind,weights=None if has_resume else 'imagenet')
    head_epochs, ft_epochs = ((STUDENT_HEAD_EPOCHS,STUDENT_FT_EPOCHS) if kind=='student'
                             else (TEACHER_HEAD_EPOCHS,TEACHER_FT_EPOCHS))
    model = run_stage(model,base,kind+'_head',False,head_epochs,1e-3)
    head_score = evaluate_frame(model,frames['val'])
    head_file = RUN/(kind+'_head.keras')
    save_model(model,head_file)
    model = run_stage(model,base,kind+'_finetune',True,ft_epochs,5e-5)
    ft_score = evaluate_frame(model,frames['val'])
    # Non peggiorare silenziosamente rispetto alla testa congelata.
    if head_score['macro_f1'] > ft_score['macro_f1']:
        model = tf.keras.models.load_model(head_file,compile=False)
        selected_score, phase = head_score,'head'
    else:
        selected_score, phase = ft_score,'finetune'
    save_model(model,RUN/(kind+'_best.keras'))
    atomic_json(RUN/'reports'/(kind+'_validation.json'),dict(phase=phase,**selected_score))
    return model


In [ ]:
# 11 — Esegue SOLO il percorso scelto. Baseline e teacher possono richiedere più sessioni.
# Le eccezioni SessionPaused sono una pausa intenzionale: non procedere alle celle dipendenti.
if MODE in {'baseline','teacher'}:
    trained_model = train_pipeline('student' if MODE=='baseline' else 'teacher')
    print('Percorso completato:', MODE)
    del trained_model
    gc.collect()
else:
    assert (RUN/'student_best.keras').exists(), 'Completa prima MODE="baseline".'
    assert (RUN/'teacher_best.keras').exists(), 'Completa prima MODE="teacher".'


## Teacher e knowledge distillation (solo `MODE="distill"`)
Il teacher è allenato esclusivamente sul train; validation sceglie i checkpoint, test non viene consultato.
Si rivalutano entrambi sulla stessa validation: il teacher deve superare la baseline di almeno 0,005 macro-F1.
Non è una prova di significatività statistica e non garantisce che la distillazione migliori lo student.

I logits train vengono salvati a blocchi con firma dei pesi, ordine degli image ID e checksum.
Se la sessione termina durante il calcolo, i blocchi completati vengono riutilizzati. Il teacher può poi uscire
dalla memoria; resta da allenare solo MobileNet. La loss è **(1−α) CE + α T² KL(teacher || student)**.


In [ ]:
# 12 — Cache dei logits del teacher e controllo del suo effettivo vantaggio.
def cached_teacher_logits(teacher, teacher_file, chunk_size=512):
    folder = RUN/'teacher_logits'
    folder.mkdir(exist_ok=True)
    signature = dict(data=DATA_HASH,weights=file_digest(teacher_file),classes=CLASSES,
                     image_size=IMG_SIZE,ids=train_frame.image_id.tolist(),chunk_size=chunk_size)
    lock_json(folder/'signature.json',signature)
    required_bytes=len(train_frame)*N_CLASSES*4
    print(f'Cache teacher locale: {required_bytes/1e9:.2f} GB; spazio Drive aggiuntivo per i blocchi compressi.')
    cache_file=LOCAL/'teacher_logits.npy'
    reclaimed=cache_file.stat().st_size if cache_file.exists() else 0
    if shutil.disk_usage(LOCAL).free+reclaimed < required_bytes+512*1024*1024:
        raise RuntimeError('Spazio locale insufficiente per i logits. Ridurre i dati in un nuovo RUN_ID o usare un disco più grande.')
    all_logits=np.lib.format.open_memmap(cache_file,mode='w+',dtype=np.float32,shape=(len(train_frame),N_CLASSES))
    for start in tqdm(range(0,len(train_frame),chunk_size),desc='Teacher logits'):
        part = train_frame.iloc[start:start+chunk_size]
        path = folder/f'logits-{start:07d}.npz'
        value = None
        if path.exists():
            try:
                with np.load(path,allow_pickle=False) as data:
                    candidate = data['logits']
                    assert np.array_equal(data['ids'],part.image_id.to_numpy().astype(str))
                    assert candidate.shape == (len(part),N_CLASSES) and np.isfinite(candidate).all()
                    assert str(data['sha']) == digest_bytes(candidate.tobytes())
                    value = candidate
            except (ValueError,KeyError,AssertionError,OSError,zipfile.BadZipFile):
                print('Ricalcolo blocco incompleto:',path.name)
        if value is None:
            value = predict_frame(teacher,part).astype(np.float32)
            assert np.isfinite(value).all()
            buf = io.BytesIO()
            np.savez_compressed(buf,logits=value,ids=part.image_id.to_numpy().astype(str),
                                sha=np.array(digest_bytes(value.tobytes())))
            atomic_bytes(path,buf.getvalue())
        all_logits[start:start+len(part)]=value
        all_logits.flush()
    return all_logits

if MODE == 'distill':
    baseline = tf.keras.models.load_model(RUN/'student_best.keras',compile=False)
    baseline_score = evaluate_frame(baseline,frames['val'])
    del baseline
    teacher = tf.keras.models.load_model(RUN/'teacher_best.keras',compile=False)
    teacher_score = evaluate_frame(teacher,frames['val'])
    print('Baseline:',baseline_score,'Teacher:',teacher_score)
    if teacher_score['macro_f1'] < baseline_score['macro_f1'] + TEACHER_MIN_GAIN:
        raise RuntimeError('Teacher non sufficientemente migliore: conserva la baseline. La distillazione non è stata avviata.')
    teacher_logits = cached_teacher_logits(teacher,RUN/'teacher_best.keras')
    del teacher
    tf.keras.backend.clear_session()
    gc.collect()
    student,base = make_model('student',weights=None)
    initial = tf.keras.models.load_model(RUN/'student_best.keras',compile=False)
    student.set_weights(initial.get_weights())
    del initial
    distilled = run_stage(student,base,'student_distill',True,KD_EPOCHS,2e-5,teacher_logits)
    save_model(distilled,RUN/'student_distilled.keras')
    kd_score = evaluate_frame(distilled,frames['val'])
    atomic_json(RUN/'reports'/'distill_validation.json',kd_score)
    print('Distillazione:',kd_score,'| baseline:',baseline_score)
    del distilled,student,teacher_logits
    gc.collect()


## Valutazione finale e scelta del modello
La scelta baseline/distillato avviene **solo sulla validation**. Il test è opzionale e va eseguito dopo avere
fissato le scelte; non usarlo per scegliere learning rate, teacher, epoca o quantizzazione.
Oltre alle metriche per immagine, si mediano le probabilità delle immagini della stessa osservazione:
questa seconda misura descrive l'uso multi-foto e non va confusa con la classificazione di una singola foto.
Il report per organo usa macro-F1 sulle specie realmente presenti in quel sottoinsieme e riporta il loro numero.

Il vecchio notebook non contiene output del test: il 52,2% citato sopra è validation, non test.
Il nuovo split potrebbe differire dal vecchio, quindi non è un confronto sperimentale controllato fra versioni.


In [ ]:
# 13 — Seleziona lo student sulla validation; nessuna selezione basata sul test.
selected_model_path = None
if (RUN/'student_best.keras').exists():
    candidates = [RUN/'student_best.keras']
    if (RUN/'student_distilled.keras').exists():
        candidates.append(RUN/'student_distilled.keras')
    scores = []
    for path in candidates:
        model = tf.keras.models.load_model(path,compile=False)
        score = evaluate_frame(model,frames['val'])
        scores.append(dict(file=path.name,**score))
        del model
    display(pd.DataFrame(scores))
    best = max(scores,key=lambda x:x['macro_f1'])
    selected_model_path = RUN/best['file']
    atomic_json(RUN/'reports'/'selected_model.json',best)
    print('Student selezionato:',selected_model_path.name)
else:
    print('Nessuna baseline completata: esegui MODE="baseline" prima di valutare lo student.')


In [ ]:
# 14 — Test finale a blocchi, anche per migliaia di classi e osservazioni.
def final_reports(model, test_frame):
    frame=test_frame.sort_values(['obs_id','image_id']).reset_index(drop=True)
    image_true,image_pred,image_hits,confidence=[],[],[],[]
    obs_true,obs_pred,obs_hits=[],[],[]
    current_id=None
    current_label=None
    sum_prob=None
    offset=0
    def close_observation():
        if sum_prob is not None:
            obs_true.append(current_label)
            obs_pred.append(int(np.argmax(sum_prob)))
            obs_hits.append(current_label in np.argpartition(sum_prob,-min(3,N_CLASSES))[-min(3,N_CLASSES):])
    for x,y in evaluation_dataset(frame):
        probs=tf.nn.softmax(model(x,training=False)).numpy()
        for prob,label in zip(probs,y.numpy()):
            row=frame.iloc[offset]
            if row.obs_id!=current_id:
                close_observation()
                current_id,current_label=row.obs_id,int(label)
                sum_prob=np.zeros(N_CLASSES,dtype=np.float64)
            assert int(label)==current_label
            sum_prob+=prob
            image_true.append(int(label));image_pred.append(int(prob.argmax()))
            image_hits.append(int(label) in np.argpartition(prob,-min(3,N_CLASSES))[-min(3,N_CLASSES):])
            confidence.append(float(prob.max()))
            offset+=1
    close_observation()
    def metrics(y,p,hits):
        return dict(top1=float(accuracy_score(y,p)),top3=float(np.mean(hits)),
            macro_f1=float(f1_score(y,p,labels=list(range(N_CLASSES)),average='macro',zero_division=0)))
    report=classification_report(image_true,image_pred,labels=list(range(N_CLASSES)),target_names=CLASSES,
                                 output_dict=True,zero_division=0)
    atomic_csv(RUN/'reports'/'test_per_species.csv',pd.DataFrame(report).T.reset_index(names='species'))
    predictions=frame[['image_id','obs_id','flora_species','organ','label']].copy()
    predictions['predicted_label']=image_pred;predictions['confidence']=confidence
    atomic_csv(RUN/'reports'/'test_predictions.csv',predictions)
    # Matrice sparsa in formato lungo: salva solo le coppie di classi realmente osservate.
    confusion=predictions.groupby(['label','predicted_label']).size().rename('count').reset_index()
    atomic_csv(RUN/'reports'/'test_confusion_pairs.csv',confusion)
    per_organ=[]
    for organ,part in predictions.groupby('organ'):
        labels=sorted(part.label.unique())
        per_organ.append(dict(organ=organ,images=len(part),species=len(labels),
            top1=float(accuracy_score(part.label,part.predicted_label)),
            macro_f1_present_species=float(f1_score(part.label,part.predicted_label,labels=labels,average='macro',zero_division=0))))
    atomic_csv(RUN/'reports'/'test_per_organ.csv',pd.DataFrame(per_organ))
    return {'image':metrics(image_true,image_pred,image_hits),'observation':metrics(obs_true,obs_pred,obs_hits)}

if RUN_FINAL_TEST:
    assert selected_model_path is not None
    model=tf.keras.models.load_model(selected_model_path,compile=False)
    results=dict(model=selected_model_path.name,data_hash=DATA_HASH,**final_reports(model,frames['test']))
    atomic_json(RUN/'reports'/'test_summary.json',results)
    print(json.dumps(results,indent=2))
    del model
else:
    print('Test finale non eseguito: RUN_FINAL_TEST=False.')


## Export TFLite per l'app
Esporta solo lo student scelto. Il modello di inferenza contiene softmax e preprocessing, senza augmentation:
input RGB 224×224 float32 in [0,255] per FP32/FP16. Per INT8 il tipo è uint8, ma va sempre letta la coppia
`scale, zero_point` dall'interprete: **non assumere che sia (1,0)**. Le uscite INT8 vanno dequantizzate.

La conversione viene confrontata con Keras su immagini validation; si salvano top-1, macro-F1, accordo delle
predizioni ed errore nelle probabilità. Questa verifica non garantisce prestazioni sul telefono.
INT8 è opzionale perché la calibrazione/conversione può costare tempo e peggiorare la qualità.


In [ ]:
# 15 — FP32/FP16 e, su richiesta, INT8; labels.json e input_contract.json accompagnano sempre i modelli.
def convert_tflite(model,variant):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if variant in {'fp16','int8'}:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    if variant == 'fp16':
        converter.target_spec.supported_types = [tf.float16]
    if variant == 'int8':
        # Campione train limitato a 300 immagini; NON copre necessariamente tutte le classi.
        calibration = pd.concat([g.sample(min(5,len(g)),random_state=SEED)
                                 for _,g in train_frame.groupby('label')],ignore_index=True)
        if len(calibration)>300:
            calibration=calibration.sample(300,random_state=SEED).reset_index(drop=True)
        def representative():
            for path in calibration.path:
                yield [decode_image(path)[None,...].numpy().astype(np.float32)]
        converter.representative_dataset = representative
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
    return converter.convert()

def lite_predict(blob,frame):
    if hasattr(tf.lite, 'Interpreter'):
        interpreter = tf.lite.Interpreter(model_content=blob)
    else:
        from ai_edge_litert.interpreter import Interpreter
        interpreter = Interpreter(model_content=blob)
    interpreter.allocate_tensors()
    inp,out = interpreter.get_input_details()[0],interpreter.get_output_details()[0]
    predictions=[]
    for path in tqdm(frame.path,desc='Verifica TFLite'):
        x = decode_image(path)[None,...].numpy()
        if np.issubdtype(inp['dtype'],np.integer):
            scale,zero = inp['quantization']
            limits=np.iinfo(inp['dtype'])
            x=np.clip(np.rint(x/scale+zero),limits.min,limits.max).astype(inp['dtype'])
        else:
            x=x.astype(inp['dtype'])
        interpreter.set_tensor(inp['index'],x)
        interpreter.invoke()
        p=interpreter.get_tensor(out['index']).astype(np.float32)
        if np.issubdtype(out['dtype'],np.integer):
            scale,zero=out['quantization']
            p=(p-zero)*scale
        predictions.append(p[0])
    return np.stack(predictions)

if EXPORT_TFLITE:
    assert selected_model_path is not None, 'Completa prima la baseline.'
    trained = tf.keras.models.load_model(selected_model_path,compile=False)
    inference = tf.keras.Model(trained.input,tf.keras.layers.Softmax(name='probabilities')(trained.output))
    folder = RUN/'exports'
    folder.mkdir(exist_ok=True)
    atomic_json(folder/'labels.json',CLASSES)
    atomic_json(folder/'input_contract.json',dict(size=[IMG_SIZE,IMG_SIZE],channels='RGB',
        resize='preserve aspect ratio with zero padding; tf.image.resize_with_pad antialias=True',
        exif='apply orientation first',float_range=[0,255],normalization='inside model',output='class probabilities',
        quantized='read scale and zero_point from interpreter',source=selected_model_path.name))
    probe = pd.concat([g.sample(min(3,len(g)),random_state=SEED)
                       for _,g in frames['val'].groupby('label')],ignore_index=True)
    if len(probe)>200:
        probe=probe.sample(200,random_state=SEED).reset_index(drop=True)
    reference=predict_frame(inference,probe)
    export_reports=[]
    for variant in ['fp32','fp16']+(['int8'] if EXPORT_INT8 else []):
        blob=convert_tflite(inference,variant)
        actual=lite_predict(blob,probe)
        assert np.isfinite(actual).all(), 'Output TFLite non finiti.'
        result=dict(variant=variant,bytes=len(blob),images=len(probe),
                    agreement=float(np.mean(actual.argmax(1)==reference.argmax(1))),
                    max_abs_error=float(np.max(np.abs(actual-reference))),**image_metrics(probe,actual))
        if variant=='fp32':
            assert np.allclose(actual,reference,atol=2e-4,rtol=2e-3), 'Export FP32 non equivalente a Keras.'
        atomic_bytes(folder/f'plant_{variant}.tflite',blob)
        export_reports.append(result)
    atomic_json(folder/'conversion_checks.json',export_reports)
    display(pd.DataFrame(export_reports))
    print('File esportati in:',folder)
else:
    print('Export non eseguito: imposta EXPORT_TFLITE=True quando hai scelto il modello.')


## Riprendere, cambiare esperimento, leggere i risultati

**Interruzione/disconnessione:** riconnetti una GPU, apri questo notebook, mantieni `RUN_ID` e parametri,
scegli lo stesso `MODE` e riesegui dall'inizio. Rimonta Drive. Immagini, split e cache teacher vengono recuperati;
ogni fase cerca l'ultimo snapshot valido. I file `.partial-*` non sono checkpoint utilizzabili.
Si riprende dall'ultimo batch salvato, non necessariamente dall'istante della disconnessione.
Lo stop manuale durante un'operazione GPU può avere granularità di un batch: per una pausa ordinata usa il budget.

**Versioni cambiate in Colab:** confronta `training_config.json`. Non cancellare questo file per aggirare il
controllo: usa le stesse versioni TensorFlow/Keras (e riavvia il runtime dopo installazione), oppure crea un nuovo
RUN_ID. Cambiare batch, classi, dati, epoche o temperatura durante la ripresa renderebbe il confronto incoerente.
I checkpoint del vecchio notebook non contengono il cursore/optimizer richiesto e non vengono importati.

**Nuovi dati / altre specie / iperparametri:** nuovo RUN_ID. Non modificare manifest o archivi di un training avviato.
Se i download falliscono prima del manifest definitivo, riesegui la cella 6; i tentativi già riusciti restano su Drive.
Assicurati di poter usare le immagini secondo la licenza e i termini di distribuzione del dataset.

**Dove guardare:**
- `reports/observation_counts.csv`, `organ_coverage.csv`: copertura reale di specie e organi;
- `reports/*-epoch-*.json`: andamento validation, numero di aggiornamenti e arresto;
- `student_best.keras`: migliore baseline; `teacher_best.keras`: teacher adattato;
- `student_distilled.keras`: candidato distillato, non automaticamente migliore;
- `reports/selected_model.json`: scelta su validation; `test_*`: test solo se richiesto;
- `exports/`: TFLite, etichette ordinate, contratto input e confronto numerico.

**Esperimenti consigliati:** completa prima la baseline; controlla gli errori per specie e organo;
poi teacher e distillazione. Se mancano organi o le immagini sono rumorose, acquisire dati migliori può essere
più utile di un teacher maggiore. Un classificatore sulle specie selezionate sceglie comunque una di esse anche per
immagini fuori insieme: la probabilità massima non costituisce un rilevatore affidabile di specie sconosciute.

**Verifica di questo file:** controlli locali su dati sintetici per split, campionamento, ripresa del checkpoint,
loss distillata e conversione sono descritti nel file di verifica allegato. Il training completo PlantCLEF,
il mount Drive e la GPU Colab devono essere eseguiti nel tuo account: il CSV e le immagini non erano allegati.


## Copertura, provenienza e verifica v3
Per la tua richiesta sono incluse anche le classi PlantCLEF aggiuntive, con presenza regionale non verificata.
La lista completa di riferimento è pubblicata anche in `data/sicilia_species_reference_2023.json` nel repository.
`taxonomy/coverage.csv` è il documento da usare per affermare cosa copre il TUO modello: non il numero totale
stampato dalla checklist. Le fotografie possono provenire anche da altre regioni, purché l'identificazione sia
corretta; per misurare l'uso in Sicilia serve inoltre un test rappresentativo di foto siciliane.

Per le lacune: cerca immagini etichettate e licenziate tramite iNaturalist/GBIF (GBIF è un aggregatore, non un
dataset di immagini già pulito). Verifica la specie, raggruppa le foto della stessa osservazione, deduplica
rispetto a PlantCLEF, conserva provenienza/licenza e rifai lo split. Per le coltivate non usare solo i filtri
iNaturalist 'wild' o 'research grade': escluderebbero molte osservazioni marcate coltivate, che richiedono
una verifica tassonomica aggiuntiva. Una singola foto non distingue sempre specie
botanicamente vicine: possono servire più organi, stagione e località. Il modello continua a essere a insieme
chiuso: una specie esclusa verrà comunque assegnata a una classe nota. La massima softmax non garantisce rifiuto
corretto delle specie sconosciute.

I controlli v2 di training/checkpoint sono ereditati. I nuovi controlli locali coprono l'estrazione dell'intera
checklist, Unicode/ibridi/sinonimi ambigui, conteggio delle osservazioni, validazione/test a blocchi e cache su disco.
Nessun conteggio di copertura PlantCLEF specifico del tuo CSV né accuratezza botanica viene dichiarato prima dell'audit reale.
Per migliaia di classi l'export INT8 usa solo un campione train limitato; la qualità sull'intero validation va
verificata separatamente prima di scegliere la quantizzazione. Il piccolo confronto di conversione non basta.
